In [21]:
import hashlib

def _stable_float(s: str) -> float:
    digest = hashlib.sha256(s.encode()).hexdigest()
    precision = (2**256)

    n = int(digest, base=16)
    return (n % precision) / precision

_stable_float("Alicia")

0.16526839816309238

In [37]:
import inspect
from types import FrameType

def _class_from_frame(frame: FrameType) -> str | None:
    self_obj = frame.f_locals.get("self")
    print(frame.f_locals.get("_ih"))
    if self_obj is not None and hasattr(self_obj, "__class__"):
        return self_obj.__class__.__name__
    cls = frame.f_locals.get("cls")
    if isinstance(cls, type):
        return cls.__name__
    return None

class test:
    def method(self):
        print(_class_from_frame(inspect.currentframe()))

test_obj = test()
test_obj.method()

print(_class_from_frame(inspect.currentframe()))

None
test
['', 'import hashlib\n\ndef _stable_float(s: str) -> float:\n    digest = hashlib.sha256(s.encode()).hexdigest()\n    print(f"Digest for \'{s}\': {digest}")\n\n    n = int(digest, 16)\n    print(f"Stable float for \'{s}\': {(n % 10_000_000)}")\n\n    return (n % 10_000_000) / 10_000_000.0\n\n_stable_float("hello world")', 'import hashlib\n\ndef _stable_float(s: str) -> float:\n    digest = hashlib.sha256(s.encode()).hexdigest()\n    print(f"Digest for \'{s}\': {digest}")\n\n    n = int(digest[:16], 16)\n    print(f"Stable float for \'{s}\': {(n % 10_000_000)}")\n\n    return (n % 10_000_000) / 10_000_000.0\n\n_stable_float("hello world")', 'import hashlib\n\ndef _stable_float(s: str) -> float:\n    digest = hashlib.sha256(s.encode()).hexdigest()\n    print(f"Digest for \'{s}\': {digest}")\n\n    n = int(digest16, 16)\n    print(f"Stable float for \'{s}\': {(n % 10_000_000)}")\n\n    return (n % 10_000_000) / 10_000_000.0\n\n_stable_float("hello world")', 'import hashlib\n\nde

In [51]:
import inspect
import dis

def example():
    a = 1
    b = 2
    return a + b

for start, end, lineno in example.__code__.co_lines():
    print(
        f"bytecode {start:>3}-{end:<3} -> source line {lineno}"
    )

print("Disassembly of example():")
dis.dis(example)

bytecode   0-4   -> source line 5
bytecode   4-8   -> source line 6
bytecode   8-16  -> source line 7
Disassembly of example():
  5           0 LOAD_CONST               1 (1)
              2 STORE_FAST               0 (a)

  6           4 LOAD_CONST               2 (2)
              6 STORE_FAST               1 (b)

  7           8 LOAD_FAST                0 (a)
             10 LOAD_FAST                1 (b)
             12 BINARY_ADD
             14 RETURN_VALUE


In [ ]:
import traceback, sys

class A(Exception): pass
class B(Exception): pass

try:
    try:
        raise A('first')
    except A as original:
        raise B('second') from original  # explicit chain
except B:
    # traceback.extract_tb gives the active exception's frames
    tb = traceback.extract_tb(sys.exc_info()[2])
    print('active frames:', len(tb))
    print('cause:', sys.exc_info()[1].__cause__)
    # The active chain
    cur = sys.exc_info()[1]
    while cur:
        print(f'  {type(cur).__name__}: {cur}')
        cur = cur.__cause__


In [9]:
import sys, traceback, linecache
from types import FrameType

def f():
    x = 45
    y = "Hello"
    return 1 / 0


try:
    f()
except:
    tb = traceback.extract_tb(sys.exc_info()[2])
    frame_info: FrameType = tb[-1]  # the leaf frame
    print('filename:', frame_info.filename)
    print('lineno:', frame_info.lineno)
    print('line:', repr(frame_info.line))
    print('function:', frame_info.name)

    start = max(1, frame_info.lineno - 2)
    end = frame_info.lineno + 2
    print('source:')
    for ln in range(start, end + 1):
        line = linecache.getline(frame_info.filename, ln).rstrip()
        marker = '  ▶' if ln == frame_info.lineno else '   '
        print(f'{marker} {ln:>3}  {line}')


    tb = sys.exc_info()[2]
    while tb:
        frame = tb.tb_frame
        print(f'=== frame: {frame.f_code.co_name} (line {tb.tb_lineno}) ===')
        # The frame's own locals
        print(f'  f_locals keys: {list(frame.f_locals.keys())}')
        # Filter to user vars (skip dunder, modules, frames)
        user_locals = {
            k: v for k, v in frame.f_locals.items()
            if not k.startswith("__")
            and not isinstance(v, (type(sys), type(f), type(tb)))
        }
        print(f'  user_locals: {user_locals}')
        tb = tb.tb_next


filename: /var/folders/jp/1hn6m8t16tq4_953r183nr0h0000gn/T/ipykernel_82484/2149838097.py
lineno: 7
line: 'return 1 / 0'
function: f
source:
      5      x = 45
      6      y = "Hello"
  ▶   7      return 1 / 0
      8  
      9  
=== frame: <module> (line 11) ===
  f_locals keys: ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'sys', 'traceback', 'linecache', 'f', 'tb', 'frame_info', 'start', 'end', 'ln', 'line', 'marker', '_i2', '_i3', 'FrameType', '__annotations__', '_i4', '_i5', '_i6', '_i7', '_i8', '_i9', 'frame']
  user_locals: {'_ih': ['', 'import sys, traceback, linecache\n\ndef f():\n    x = 45\n    y = "Hello"\n    return 1 / 0\n\n\ntry:\n    f()\nexcept:\n    tb = traceback.extract_tb(sys.exc_info()[2])\n    tb = traceback.extract_tb(sys.exc_info()[2])\n    frame_info = tb[-1]  # the leaf 

In [34]:
import sys, traceback

def g():
    a = 10
    b = 20
    return f()

def f():
    x = 42
    y = 'hello'
    d = {'nested': [1, 2, 3]}
    return 1/0

try:
    g()
except ZeroDivisionError:
    exc_type, exc_val, exc_tb = sys.exc_info()
    # Use traceback.extract_tb to get a list of FrameSummary objects
    # in source order.
    summaries = traceback.extract_tb(exc_tb)
    print('=== extract_tb (in source order) ===')
    for s in summaries:
        print(f'  {s.filename.rsplit("/", 1)[-1]}:{s.lineno}  in {s.name}')
        print(f'    line: {s.line!r}')
        if s.name == 'f':
            # Show what extract_tb sees as locals — it doesn't, but
            # we can find the frame via exc_tb walk.
            print(f'    f_locals (not available on summary)')

    # Now walk exc_tb to get the actual frames
    print('=== walking exc_tb for f_locals ===')
    cur = exc_tb
    while cur is not None:
        f = cur.tb_frame
        print(f'  frame: {f.f_code.co_name} at line {cur.tb_lineno}')
        print(f'    f_locals: {list(f.f_locals.keys())}')
        cur = cur.tb_next


=== extract_tb (in source order) ===
  2534994719.py:15  in <module>
    line: 'g()'
  2534994719.py:6  in g
    line: 'return f()'
  2534994719.py:12  in f
    line: 'return 1/0'
    f_locals (not available on summary)
=== walking exc_tb for f_locals ===
  frame: <module> at line 15
    f_locals: ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'sys', 'traceback', 'linecache', 'f', 'tb', 'frame_info', 'start', 'end', 'ln', 'line', 'marker', '_i2', '_i3', 'FrameType', '__annotations__', '_i4', '_i5', '_i6', '_i7', '_i8', '_i9', 'frame', 'user_locals', '_i10', 'exc_type', 'exc_val', 'exc_tb', 'summaries', 's', 'cur', '_i11', 'g', '_i12', '_i13', '_i14', '_i15', '_i16', '_i17', '_i18', '_i19', '_i20', '_i21', '_i22', '_i23', '_i24', '_i25', 'frames', 'i', '_i26', '_i27', '_i28', '_i29', '_i30', '_i31', 

In [ ]:
import sys, traceback

def g():
    a = 10
    b = 20
    return f()

def f():
    x = 42
    y = 'hello'
    d = {'nested': [1, 2, 3]}
    return 1/0

try:
    g()
except ZeroDivisionError:
    exc_type, exc_val, exc_tb = sys.exc_info()
    summaries = traceback.extract_tb(exc_tb)

    frames = []
    cur = exc_tb
    while cur is not None:
        frames.append(cur.tb_frame)
        cur = cur.tb_next

    print('=== frames order (innermost first) ===')
    for i, f in enumerate(frames):
        print(f'  [{i}] {f.f_code.co_name} line {f.f_lineno}, locals={list(f.f_locals.keys())}')

    print('=== summaries order ===')
    for i, s in enumerate(summaries):
        print(f'  [{i}] {s.name} line {s.lineno}')

    max_width = max(len(frame.f_code.co_name) for frame in frames)
    print(f'max_width: {max_width}')

=== frames order (innermost first) ===
  [0] <module> line 28, locals=['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'sys', 'traceback', 'linecache', 'f', 'tb', 'frame_info', 'start', 'end', 'ln', 'line', 'marker', '_i2', '_i3', 'FrameType', '__annotations__', '_i4', '_i5', '_i6', '_i7', '_i8', '_i9', 'frame', 'user_locals', '_i10', 'exc_type', 'exc_val', 'exc_tb', 'summaries', 's', 'cur', '_i11', 'g', '_i12', '_i13', '_i14', '_i15', '_i16', '_i17', '_i18', '_i19', '_i20', '_i21', '_i22', '_i23', '_i24', '_i25', 'frames', 'i', '_i26', '_i27', '_i28', '_i29', '_i30', '_i31', '_i32', '_i33', '_i34', '_i35', 'max_width', '_i36', '_i37']
  [1] g line 6, locals=['a', 'b']
  [2] f line 12, locals=['x', 'y', 'd']
=== summaries order ===
  [0] <module> line 15
  [1] g line 6
  [2] f line 12
max_width: 8
